# Needlet / Harmonic ILC Compton-$y$ on FLAMINGO mocks (pyILC)

McCarthy & Hill (2024) style undeprojected Compton-$y$ ILC
([arXiv:2307.01043](https://doi.org/10.48550/arxiv.2307.01043)) on **FLAMINGO mock**
skies (CMB + tSZ + CIB) **plus Planck FFP10 noise splits** at 100/143/353 GHz.

Settings:
- `ELLMAX: 3000`, `N_side: 2048`
- Channel beams 9.66′ / 7.22′ / 4.92′; common ILC beam 5′ (spectra beam-deconvolved)
- `ILC_preserved_comp: tSZ`, `N_deproj: 0`
- Independent noise realizations: FFP10 `mc_00000` / `mc_00001`

**Environment:** `source ~/envs/cosmo_env/bin/activate`

In [ ]:
import os
from pathlib import Path

REPO = Path('..').resolve()
os.chdir(REPO)
os.environ.setdefault('PYILC_BACKEND', 'numba')
print('cwd', Path.cwd())
print('backend', os.environ['PYILC_BACKEND'])

## 1. Prepare beam-smoothed coadds + FFP10 noise splits (nside=2048)

In [ ]:
%run -i scripts/prepare_mock_ilc_inputs.py --nside 2048

## 2. HILC on noise split 0 and 1 (ELLMAX=3000)

In [ ]:
%run -i scripts/run_pyilc_y.py configs/hilc_y_flamingo_noise_split0.yml --backend numba
%run -i scripts/run_pyilc_y.py configs/hilc_y_flamingo_noise_split1.yml --backend numba

## 3. Optional: paper-style NILC (Gaussian needlets)

In [ ]:
# Heavier than HILC; intermediates under ~/cosmology_data/flamingo_ilc/nilc_output_noise_split0/
# %run -i scripts/run_pyilc_y.py configs/nilc_y_flamingo_noise_split0.yml --backend numba

## 4. Validate (beam-deconvolved $C_\ell$ to $\ell=3000$)

In [ ]:
from pathlib import Path
base = Path.home() / 'cosmology_data/flamingo_ilc'
y0 = base / 'hilc_output_noise_split0/flamingo_needletILCmap_component_tSZ_hilc_y_noise_split0.fits'
y1 = base / 'hilc_output_noise_split1/flamingo_needletILCmap_component_tSZ_hilc_y_noise_split1.fits'
truth = base / 'inputs_nside2048_noise/compton_y_nside2048.fits'
assert y0.is_file(), y0
%run -i scripts/validate_ymap_vs_truth.py --ymap {y0} --ymap-split1 {y1} --truth {truth} --figures-dir figures --lmax 3000 --ilc-beam-fwhm-arcmin 5.0